# PDI — Fase 1 | Inventário, validação e Ground Truth (4 datasets)

**Responsável pela Fase 1:** Vivian Beatriz · **Consolidação:** equipe PDI · **Data:** 25/11/2026

**Propósito:** executar do início ao fim no Google Colab, lendo versões já revisadas no Google Drive, sem modificar nenhum ZIP. Cada dataset mantém IDs de classe independentes. O Ground Truth é derivado das anotações existentes e não foi conferido manualmente em todas as imagens.

**Versões selecionadas:** parafusos `revisado.zip` (463/3.230); porcas/multiclasse `filtrado.zip` (107/2.832); arruelas `deduplicado.zip` (402/3.949); vergalhões `original.zip` (3/26, somente treino). Os números são referências para detectar alterações acidentais.

**Escopo da fase:** organização, avaliação estrutural de anotações YOLO (detecção ou segmentação), auditoria de duplicatas exatas, geração de Ground Truth e rastreabilidade. Não inclui treinamento, métricas preditivas ou consolidação de classes entre datasets.


## 1. Ambiente e Google Drive
Execute as células na ordem no Google Colab e autorize o acesso ao Drive.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import hashlib, json, math, re, shutil, zipfile
import pandas as pd
import yaml
import numpy as np
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive')
print('Bibliotecas carregadas e Drive conectado.')

## 2. Configuração e seleção das versões definitivas
Não use automaticamente `original.zip` quando uma versão revisada já foi aprovada. Se faltar um ZIP, a execução interrompe com mensagem explícita.

In [ ]:
# Configuração: edite RAIZ_DRIVE somente se a pasta estiver em outro local.
RAIZ_DRIVE = Path('/content/drive/MyDrive/PDI_Contagem_Objetos')
PASTA_DATASETS = RAIZ_DRIVE / 'datasets'

# Versões aprovadas na auditoria da equipe (não substituir silenciosamente pelo original).
VERSOES_FINAIS = {
    'parafusos/roboflow_screws': 'revisado.zip',
    'porcas/nutsnbolts': 'filtrado.zip',
    'arruelas/washers_hardee': 'deduplicado.zip',
    'vergalhoes/rebar_counting_v1': 'original.zip',
}

# None = os quatro datasets. Para executar apenas um: {'porcas/nutsnbolts'}
SELECIONAR_DATASETS = None

if not PASTA_DATASETS.is_dir():
    raise FileNotFoundError(f'Pasta do Drive não encontrada: {PASTA_DATASETS}')

ids = set(VERSOES_FINAIS) if SELECIONAR_DATASETS is None else set(SELECIONAR_DATASETS)
invalidos = ids - set(VERSOES_FINAIS)
if invalidos:
    raise ValueError(f'Datasets não cadastrados: {sorted(invalidos)}')

catalogo = {}
for dataset_id in sorted(ids):
    categoria, nome = dataset_id.split('/', 1)
    pasta = PASTA_DATASETS / categoria / nome
    nome_zip = VERSOES_FINAIS[dataset_id]
    origem = pasta / 'original.zip'
    fonte = pasta / nome_zip
    if not origem.is_file():
        raise FileNotFoundError(f'Original obrigatório não encontrado: {origem}')
    if not fonte.is_file():
        raise FileNotFoundError(f'Versão final obrigatória não encontrada: {fonte}')
    catalogo[dataset_id] = dict(arquivo=fonte, original=origem,
                               versao=fonte.stem, categoria=categoria, nome=nome)

print('Datasets cadastrados e versões selecionadas:')
for dataset_id, d in catalogo.items():
    print(f'  {dataset_id} — {d["arquivo"].name} ({d["arquivo"].stat().st_size / 1024**2:.2f} MB)')


## 3. Validação de imagens e anotações
Detecção YOLO: `classe cx cy w h`. Segmentação: `classe x1 y1 ...`, pelo menos três vértices. Uma linha equivale a uma instância anotada. Hash SHA-256 detecta somente duplicatas idênticas, não imagens transformadas.

Esta etapa não altera imagens nem labels.

In [ ]:
EXTENSOES_IMAGEM = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
COLUNAS_PROBLEMAS = ['imagem', 'anotacao', 'linha', 'problema', 'conteudo']
SPLITS = {'train', 'valid', 'test'}

def validar_linha(valores, classes):
    try:
        if not valores:
            return None, None, 'Linha vazia'
        classe = int(valores[0])
        if classe not in classes:
            return None, None, 'ID de classe inválido'
        coords = [float(v) for v in valores[1:]]
        if not all(math.isfinite(v) for v in coords):
            return None, None, 'Coordenadas não finitas'
        if len(coords) == 4:
            x, y, w, h = coords
            if not (0 < w <= 1 and 0 < h <= 1):
                return None, None, 'Largura/altura inválida'
            eps = 1e-8
            if not (0-eps <= x-w/2 <= 1+eps and 0-eps <= x+w/2 <= 1+eps
                    and 0-eps <= y-h/2 <= 1+eps and 0-eps <= y+h/2 <= 1+eps):
                return None, None, 'Caixa fora da imagem'
            return classe, 'deteccao', None
        if len(coords) >= 6 and len(coords) % 2 == 0:
            if not all(0 <= c <= 1 for c in coords):
                return None, None, 'Coordenada de polígono fora de [0,1]'
            xs, ys = coords[::2], coords[1::2]
            if max(xs) - min(xs) <= 1e-8 or max(ys) - min(ys) <= 1e-8:
                return None, None, 'Polígono degenerado (sem área delimitadora)'
            return classe, 'segmentacao', None
        return None, None, 'Formato de anotação não reconhecido'
    except (ValueError, IndexError, TypeError):
        return None, None, 'Classe ou coordenadas inválidas'

def extrair_zip_seguro(arquivo_zip, destino):
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True, exist_ok=True)
    raiz = destino.resolve()
    with zipfile.ZipFile(arquivo_zip) as z:
        for info in z.infolist():
            membro = Path(info.filename)
            saida = (destino / membro).resolve()
            if not saida.is_relative_to(raiz) or info.filename.startswith('/'):
                raise ValueError(f'Caminho inseguro dentro do ZIP: {info.filename}')
        z.extractall(destino)

def buscar_configuracao(raiz):
    candidatos = sorted(p for p in raiz.rglob('data.yaml') if p.is_file())
    if len(candidatos) != 1:
        raise ValueError(f'Esperado um data.yaml, encontrados {len(candidatos)}')
    cfg = yaml.safe_load(candidatos[0].read_text(encoding='utf-8')) or {}
    nomes = cfg.get('names')
    if isinstance(nomes, list):
        classes = {i: str(n) for i, n in enumerate(nomes)}
    elif isinstance(nomes, dict):
        classes = {int(k): str(v) for k, v in nomes.items()}
    else:
        raise ValueError('data.yaml sem names em formato lista/dicionário')
    if not classes or len(classes) != len(set(classes)):
        raise ValueError('Lista de classes inválida')
    return classes, candidatos[0]

def conjunto_da_imagem(caminho_relativo):
    encontrados = [p for p in caminho_relativo.parts if p in SPLITS]
    return encontrados[0] if len(encontrados) == 1 else 'desconhecido'

def caminho_label(imagem):
    partes = list(imagem.parts)
    indices = [i for i, p in enumerate(partes) if p == 'images']
    if len(indices) != 1:
        raise ValueError(f'Caminho sem uma pasta images/ inequívoca: {imagem}')
    partes[indices[0]] = 'labels'
    return Path(*partes).with_suffix('.txt')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with caminho.open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

def nome_base_sem_augmentacao(caminho):
    # Ex.: nome_jpg.rf.abcdef.jpg -> nome_jpg (indica possível origem em comum)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', caminho.stem, flags=re.IGNORECASE)

def validar_dataset(raiz, dataset_id):
    classes, config = buscar_configuracao(raiz)
    imagens = sorted(p for p in raiz.rglob('*') if p.is_file()
                     and p.suffix.lower() in EXTENSOES_IMAGEM and 'images' in p.parts)
    if not imagens:
        raise ValueError('Nenhuma imagem encontrada em images/')
    registros, problemas = [], []
    por_formato, por_classe = Counter(), Counter()
    hashes, origens = defaultdict(list), defaultdict(list)
    labels_esperadas = set()
    for img in imagens:
        rel = img.relative_to(raiz)
        split = conjunto_da_imagem(rel)
        lab = caminho_label(img)
        labels_esperadas.add(lab.resolve())
        rel_lab = str(lab.relative_to(raiz))
        contagem = Counter()
        n_invalidas = 0
        if split == 'desconhecido':
            problemas.append({'imagem':str(rel),'anotacao':rel_lab,'linha':None,
                              'problema':'Conjunto train/valid/test desconhecido','conteudo':''})
        if not lab.is_file():
            problemas.append({'imagem':str(rel),'anotacao':rel_lab,'linha':None,
                              'problema':'Arquivo de anotação ausente','conteudo':''})
        else:
            for numero, linha in enumerate(lab.read_text(encoding='utf-8-sig').splitlines(),1):
                valores = linha.strip().split()
                if not valores:
                    continue
                cid, formato, erro = validar_linha(valores, classes)
                if erro:
                    n_invalidas += 1
                    problemas.append({'imagem':str(rel),'anotacao':rel_lab,'linha':numero,
                                      'problema':erro,'conteudo':linha.strip()})
                else:
                    contagem[cid] += 1
                    por_classe[cid] += 1
                    por_formato[formato] += 1
        r = {'dataset_id':dataset_id, 'imagem':str(rel), 'conjunto':split,
             'possui_anotacao':lab.is_file(),'quantidade_valida':sum(contagem.values()),
             'anotacoes_invalidas':n_invalidas,
             'revisao_pendente':not lab.is_file() or n_invalidas>0 or split=='desconhecido'}
        for cid, nome in sorted(classes.items()):
            r[f'classe_{cid}_{nome}'] = contagem[cid]
        registros.append(r)
        hashes[sha256_arquivo(img)].append({'imagem':str(rel),'conjunto':split})
        origens[nome_base_sem_augmentacao(img)].append({'imagem':str(rel),'conjunto':split})
    labels_reais = {p.resolve() for p in raiz.rglob('*.txt') if 'labels' in p.parts}
    for sobra in sorted(labels_reais - labels_esperadas):
        problemas.append({'imagem':'','anotacao':str(sobra.relative_to(raiz)), 'linha':None,
                          'problema':'Anotação sem imagem correspondente','conteudo':''})
    duplicatas = []
    duplicatas_exatas = []
    for hash_img, itens in hashes.items():
        if len(itens) > 1:
            duplicatas_exatas.append({'sha256': hash_img, 'quantidade': len(itens),
                                      'copias_excedentes': len(itens)-1,
                                      'entre_splits': len({i['conjunto'] for i in itens}) > 1,
                                      'imagens': ' | '.join(i['imagem'] for i in itens)})
    for criterio, grupos in [('sha256_identico',hashes),('mesmo_nome_origem',origens)]:
        for identificador, itens in grupos.items():
            if len({i['conjunto'] for i in itens}) > 1:
                duplicatas.append({'criterio':criterio,'chave':identificador,
                                   'quantidade':len(itens),
                                   'conjuntos':' | '.join(sorted({i['conjunto'] for i in itens})),
                                   'imagens':' | '.join(i['imagem'] for i in itens)})
    gt = pd.DataFrame(registros).sort_values(['conjunto','imagem']).reset_index(drop=True)
    erros = pd.DataFrame(problemas, columns=COLUNAS_PROBLEMAS)
    dup = pd.DataFrame(duplicatas, columns=['criterio','chave','quantidade','conjuntos','imagens'])
    dup_exatas = pd.DataFrame(duplicatas_exatas, columns=['sha256','quantidade','copias_excedentes','entre_splits','imagens'])
    resumo = {
        'dataset_id':dataset_id,'imagens':len(gt),
        'distribuicao':{str(k):int(v) for k,v in gt['conjunto'].value_counts().items()},
        'classes':{str(k):v for k,v in sorted(classes.items())},
        'instancias_validas':int(gt['quantidade_valida'].sum()),
        'anotacoes_por_formato':dict(por_formato),
        'objetos_por_classe':{str(k):int(v) for k,v in sorted(por_classe.items())},
        'ocorrencias_problemas':len(problemas),
        'imagens_revisao_pendente':int(gt['revisao_pendente'].sum()),
        'grupos_suspeitos_entre_splits':len(duplicatas),
        'grupos_duplicatas_exatas':len(duplicatas_exatas),
        'copias_exatas_excedentes':sum(g['copias_excedentes'] for g in duplicatas_exatas),
        'gerado_em_utc':datetime.now(timezone.utc).isoformat(),
        'observacao':'Validação estrutural: contagens das anotações; completude visual não verificada. '
                     'Mesmo nome de origem é indício, não prova, de vazamento de dados.'}
    return gt, erros, dup, dup_exatas, resumo

## 4. Processamento dos quatro datasets
Gera uma pasta única por execução em `resultados/fase1_multidataset/`. Salva Ground Truth, problemas, suspeitas de origem, duplicatas exatas, resumo e manifesto com SHA-256 da fonte.

In [ ]:
EXECUCAO_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PASTA_EXECUCAO = RAIZ_DRIVE / 'resultados' / 'fase1_multidataset' / EXECUCAO_ID
PASTA_EXECUCAO.mkdir(parents=True, exist_ok=False)

resumos_execucao = []
resultados_em_memoria = {}
for dataset_id, d in catalogo.items():
    print(f'\n{"="*65}\n{dataset_id} | {d["arquivo"].name}')
    local = Path('/content/pdi_fase1_final') / d['categoria'] / d['nome']
    saida = PASTA_EXECUCAO / d['categoria'] / d['nome']
    saida.mkdir(parents=True, exist_ok=True)
    try:
        extrair_zip_seguro(d['arquivo'], local)
        gt, erros, dup, dup_exatas, resumo = validar_dataset(local, dataset_id)
        resumo.update(arquivo_fonte=str(d['arquivo']),
                      versao_fonte=d['versao'],
                      sha256_zip_fonte=sha256_arquivo(d['arquivo']))
        distribuicao = resumo['distribuicao']
        if dataset_id == 'vergalhoes/rebar_counting_v1':
            resumo['status_qualidade'] = 'exploratorio_sem_validacao_ou_teste'
            resumo['limitacao'] = ('Apenas 3 imagens em train e 26 rótulos na classe rebar edge; '
                                  'não permite avaliação estatística confiável. É necessário ampliar o dataset.')
        elif len(erros) or int(resumo['imagens_revisao_pendente']) > 0:
            resumo['status_qualidade'] = 'revisao_estrutural_pendente'
        elif len(dup_exatas) > 0 or any(x['criterio'] == 'sha256_identico' for _, x in dup.iterrows()):
            resumo['status_qualidade'] = 'revisar_duplicatas_exatas'
        elif distribuicao.get('train', 0) == 0 or distribuicao.get('valid', 0) == 0 or distribuicao.get('test', 0) == 0:
            resumo['status_qualidade'] = 'split_incompleto'
        else:
            resumo['status_qualidade'] = 'validado_estruturalmente'
        if len(dup):
            resumo['alerta_origem'] = ('Há grupos com nomes de origem compartilhados entre splits; '
                                      'isso não comprova duplicidade, requer análise visual.')
        resumo['status_execucao'] = 'analisado'
        gt.to_csv(saida / 'ground_truth.csv', index=False, encoding='utf-8-sig')
        erros.to_csv(saida / 'problemas_anotacoes.csv', index=False, encoding='utf-8-sig')
        dup.to_csv(saida / 'suspeitas_entre_splits.csv', index=False, encoding='utf-8-sig')
        dup_exatas.to_csv(saida / 'duplicatas_exatas.csv', index=False, encoding='utf-8-sig')
        (saida / 'resumo.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding='utf-8')
        resultados_em_memoria[dataset_id] = dict(ground_truth=gt, problemas=erros,
                                                 duplicatas=dup, duplicatas_exatas=dup_exatas,
                                                 resumo=resumo)
        print('Imagens:', resumo['imagens'], 'Objetos:', resumo['instancias_validas'])
        print('Splits:', distribuicao, 'Classes:', resumo['classes'])
        print('Problemas:', len(erros), 'Duplicatas exatas:', len(dup_exatas))
        print('Status:', resumo['status_qualidade'])
    except Exception as exc:
        resumo = {'dataset_id':dataset_id, 'status_execucao':'erro',
                  'status_qualidade':'erro_na_execucao', 'erro':str(exc),
                  'arquivo_fonte':str(d['arquivo']), 'versao_fonte':d['versao']}
        (saida / 'erro_execucao.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding='utf-8')
        print('ERRO:', str(exc))
    resumos_execucao.append(resumo)

consolidado = pd.DataFrame([{
    'dataset_id': r['dataset_id'], 'versao_fonte':r.get('versao_fonte',''),
    'status_execucao':r['status_execucao'], 'status_qualidade':r.get('status_qualidade',''),
    'imagens':r.get('imagens'), 'instancias_validas':r.get('instancias_validas'),
    'problemas':r.get('ocorrencias_problemas'),
    'imagens_revisao_pendente':r.get('imagens_revisao_pendente'),
    'duplicatas_exatas':r.get('grupos_duplicatas_exatas'),
    'suspeitas_entre_splits':r.get('grupos_suspeitos_entre_splits'),
    'erro':r.get('erro','')
} for r in resumos_execucao])
consolidado.to_csv(PASTA_EXECUCAO/'consolidado.csv', index=False, encoding='utf-8-sig')
(PASTA_EXECUCAO/'manifesto_execucao.json').write_text(json.dumps(resumos_execucao,ensure_ascii=False,indent=2), encoding='utf-8')
print('\n=== CONSOLIDADO FINAL — FASE 1 ===')
display(consolidado)
print('Resultados:', PASTA_EXECUCAO)
print('Datasets originais e versões finais não foram alterados.')


## 5. Conferência dos critérios de aceitação
Os valores esperados vêm dos relatórios anteriores da equipe. A ausência de problemas geométricos é necessária, mas **não atesta completude de todas as anotações visíveis**. O dataset de vergalhões é apenas exploratório e não está pronto para avaliação de modelo.

In [ ]:
# Critérios de aceitação estrutural por dataset (resultados confirmados na auditoria de 25/11/2026).
# São verificações de regressão do inventário: não substituem a inspeção visual.
REFERENCIA = {
    'parafusos/roboflow_screws': (463, 3230),
    'porcas/nutsnbolts': (107, 2832),
    'arruelas/washers_hardee': (402, 3949),
    'vergalhoes/rebar_counting_v1': (3, 26),
}

avaliacoes = []
for dataset_id, d in resultados_em_memoria.items():
    resumo = d['resumo']
    esperado = REFERENCIA[dataset_id]
    contagens_ok = (resumo['imagens'], resumo['instancias_validas']) == esperado
    zero_problemas = resumo['ocorrencias_problemas'] == 0
    zero_duplicatas_exatas = resumo['grupos_duplicatas_exatas'] == 0
    avaliacoes.append({'dataset_id':dataset_id,
                       'imagens_esperadas':esperado[0], 'objetos_esperados':esperado[1],
                       'contagens_conferem':contagens_ok,
                       'sem_problemas_estruturais':zero_problemas,
                       'sem_duplicatas_exatas':zero_duplicatas_exatas,
                       'status':resumo['status_qualidade']})

criterios = pd.DataFrame(avaliacoes)
display(criterios)
criterios.to_csv(PASTA_EXECUCAO/'criterios_aceitacao.csv', index=False, encoding='utf-8-sig')

if len(resultados_em_memoria) != len(catalogo):
    raise RuntimeError('Um ou mais datasets falharam na execução.')
if not all(criterios['contagens_conferem'] & criterios['sem_problemas_estruturais'] & criterios['sem_duplicatas_exatas']):
    raise AssertionError('A auditoria encontrou divergências. Confira os arquivos de relatório.')
print('\nInventário, geometria dos rótulos e duplicatas binárias: verificações aprovadas.')
print('ATENÇÃO: vergalhões continuam exploratórios; não possuem valid/test.')
print('ATENÇÃO: verificar conteúdo visual, vazamento por imagens semelhantes e equivalência de classes antes de treinar.')


## 6. Histórico das decisões, limitações e próxima fase

| Dataset | Fonte utilizada | Decisão auditada | Limitação |
|---|---|---|---|
| Parafusos | `revisado.zip` | 2 caixas limitadas às bordas e 3 rótulos descartados na versão derivada | Contagem baseada nos rótulos, não em inspeção exaustiva |
| NutsNBolts | `filtrado.zip` | 23 imagens com 32 rótulos problemáticos excluídas somente da versão filtrada | Conjunto de validação/teste com apenas 5 imagens cada; três classes `bolt`, `nut`, `washer` |
| Washers Hardee | `deduplicado.zip` | Remoção de 3 cópias redundantes iniciais e de outras 331 duplicatas exatas | SHA-256 não detecta imagens apenas semelhantes; verificar licença e diversidade |
| Vergalhões | `original.zip` | Manter três imagens com 26 rótulos | Todas em `train`, sem `valid` ou `test`; classe `rebar edge`, não necessariamente toda a barra |

**Auditoria histórica:** os relatórios de revisão/filtragem permanecem em `resultados/auditorias/` e em `resultados/fase1_multidataset/validacao_final/`. Este notebook **não repete as exclusões** nem sobrescreve os ZIPs.

**Antes de treinar:** padronizar nomenclatura de classes e formato dos rótulos, conferir semelhança visual entre splits, validar licenças e coletar mais imagens de vergalhões. A Fase 2 (pré-processamento) deve usar as mesmas versões finais e conservar vínculos imagem ↔ anotação.


## 7. Métricas para uso futuro
A função abaixo é disponibilizada para a etapa de avaliação; não mede a qualidade dos dados nem gera predições nesta fase.

In [ ]:
def calcular_metricas(resultados):
    colunas = {'quantidade_real', 'quantidade_detectada'}
    if not colunas.issubset(resultados.columns):
        raise ValueError(f'Colunas obrigatórias: {sorted(colunas)}')
    if resultados.empty:
        raise ValueError('Nenhuma imagem disponível')
    if 'revisao_pendente' in resultados.columns and resultados['revisao_pendente'].fillna(False).any():
        raise ValueError('Existem imagens com Ground Truth pendente de revisão')
    real = pd.to_numeric(resultados['quantidade_real'],errors='raise')
    pred = pd.to_numeric(resultados['quantidade_detectada'],errors='raise')
    if real.isna().any() or pred.isna().any():
        raise ValueError('As contagens não podem conter valores ausentes')
    erros = real - pred
    return {'MAE':float(erros.abs().mean()),
            'RMSE':float(np.sqrt((erros**2).mean())),
            'Acuracia_exata (%)':float((erros == 0).mean() * 100)}

print('Função de métricas carregada. Sem treinamento nesta fase.')